In [90]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from torchvision import models as tv_models
import warnings
import torch
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import TensorDataset, DataLoader
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import torch.nn as nn
warnings.filterwarnings('ignore')

In [91]:
train_df = pd.read_csv(r'C:\Users\Rama\Desktop\SPRING-Junior\ANN\Kaggle\ann-spr-26-nu\multimodal_competition\processed\train.csv')
test_df = pd.read_csv(r'C:\Users\Rama\Desktop\SPRING-Junior\ANN\Kaggle\ann-spr-26-nu\multimodal_competition\processed\test.csv')
data_dict = pd.read_csv(r'C:\Users\Rama\Desktop\SPRING-Junior\ANN\Kaggle\ann-spr-26-nu\multimodal_competition\processed\data_dictionary.csv')
image_folder = r'C:\Users\Rama\Desktop\SPRING-Junior\ANN\Kaggle\ann-spr-26-nu\multimodal_competition\processed\images_224'

In [92]:
train_df['text'] = train_df['title'] + " " + train_df['blurb']
test_df['text'] = test_df['title'] + " " + test_df['blurb']

train_df['text_length'] = train_df['text'].str.len()
train_df['title_length'] = train_df['title'].str.len()
train_df['blurb_length'] = train_df['blurb'].str.len()

test_df['text_length'] = test_df['text'].str.len()
test_df['title_length'] = test_df['title'].str.len()
test_df['blurb_length'] = test_df['blurb'].str.len()

# Define feature columns
numerical_cols = [
    'goal_usd', 'duration_days', 'launch_hour_local',
    'creator_num_prior_projects', 'creator_prior_success_rate',
    'num_reward_tiers', 'description_length'
]

categorical_cols = [
    'category_main', 'category_sub', 'country', 'currency',
    'launch_month', 'launch_weekday', 'has_video'
]

numerical_cols.extend(['text_length', 'title_length', 'blurb_length'])

# ============================================
# 2. TABULAR FEATURES (Use XGBoost later)
# ============================================

print("\n" + "="*50)
print("Processing Tabular Features")
print("="*50)

# Encode categoricals
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    label_encoders[col] = le
    test_df[col] = test_df[col].astype(str).map(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

# Scale numericals
scaler = StandardScaler()
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

X_tabular_train = train_df[categorical_cols + numerical_cols].values
X_tabular_test = test_df[categorical_cols + numerical_cols].values
y = train_df['target_band'].values

print(f"✓ Tabular features: {X_tabular_train.shape}")


Processing Tabular Features
✓ Tabular features: (8870, 17)


In [93]:

print("\n" + "="*50)
print("Extracting Image Features (EfficientNet)")
print("="*50)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load EfficientNet
cnn = tv_models.efficientnet_b0(pretrained=True)
cnn.classifier = nn.Identity()  # Remove classification layer
cnn.to(device)
cnn.eval()

# Image transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Fix image paths
image_folder = Path(r'C:\Users\Rama\Desktop\SPRING-Junior\ANN\Kaggle\ann-spr-26-nu\multimodal_competition\processed\images_224')
train_df['full_image_path'] = train_df['image_path'].apply(lambda x: str(image_folder / Path(x).name))
test_df['full_image_path'] = test_df['image_path'].apply(lambda x: str(image_folder / Path(x).name))

def extract_image_features(image_paths, batch_size=64):
    """Extract EfficientNet features from images"""
    features = []
    n_images = len(image_paths)
    
    for i in tqdm(range(0, n_images, batch_size), desc="Extracting image features"):
        batch_paths = image_paths[i:i+batch_size]
        batch_tensors = []
        
        for img_path in batch_paths:
            try:
                img = Image.open(img_path).convert('RGB')
                img_tensor = transform(img)
                batch_tensors.append(img_tensor)
            except:
                batch_tensors.append(torch.zeros(3, 224, 224))
        
        batch = torch.stack(batch_tensors).to(device)
        with torch.no_grad():
            batch_features = cnn(batch).squeeze().cpu().numpy()
        
        if batch_features.ndim == 1:
            batch_features = batch_features.reshape(1, -1)
        features.extend(batch_features)
    
    return np.array(features)

# Extract features
train_image_features = extract_image_features(train_df['full_image_path'].values)
test_image_features = extract_image_features(test_df['full_image_path'].values)
print(f"✓ Image features shape: {train_image_features.shape}")


Extracting Image Features (EfficientNet)
Using device: cuda


Extracting image features: 100%|██████████| 49/49 [00:15<00:00,  3.07it/s]

✓ Image features shape: (8870, 1280)


In [94]:
print("\n" + "="*50)
print("Training LSTM for Text Features")
print("="*50)

# Tokenization functions
def create_vocabulary(texts, max_words=20000):
    word_counts = Counter()
    for text in texts:
        words = text.lower().split()
        word_counts.update(words)
    
    most_common = word_counts.most_common(max_words - 2)
    word2idx = {word: idx+2 for idx, (word, _) in enumerate(most_common)}
    word2idx['<PAD>'] = 0
    word2idx['<UNK>'] = 1
    
    print(f"  Total unique words: {len(word_counts)}")
    print(f"  Keeping top: {len(most_common)} words")
    return word2idx

def text_to_sequences(texts, word2idx, max_len=100):
    sequences = []
    for text in texts:
        words = text.lower().split()[:max_len]
        seq = [word2idx.get(word, word2idx['<UNK>']) for word in words]
        if len(seq) < max_len:
            seq += [0] * (max_len - len(seq))
        else:
            seq = seq[:max_len]
        sequences.append(seq)
    return np.array(sequences)

# Create vocabulary
all_texts = train_df['text'].tolist() + test_df['text'].tolist()
word2idx = create_vocabulary(all_texts, max_words=20000)
vocab_size = len(word2idx)
print(f"✓ Vocabulary size: {vocab_size}")

# Convert to sequences
max_len = 100
train_sequences = text_to_sequences(train_df['text'].tolist(), word2idx, max_len)
test_sequences = text_to_sequences(test_df['text'].tolist(), word2idx, max_len)
print(f"✓ Sequences shape: {train_sequences.shape}")

# LSTM Model for training
class TextLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, num_classes=5):
        super(TextLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2,
                           batch_first=True, dropout=0.3, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        hidden_combined = torch.cat((hidden[-2], hidden[-1]), dim=1)
        output = self.fc(hidden_combined)
        return output

# Prepare data for LSTM training
X_train_seq, X_val_seq, y_train_seq, y_val_seq = train_test_split(
    train_sequences, y, test_size=0.2, random_state=42, stratify=y
)

# Create DataLoaders
batch_size = 128
train_dataset = TensorDataset(torch.LongTensor(X_train_seq), torch.LongTensor(y_train_seq))
val_dataset = TensorDataset(torch.LongTensor(X_val_seq), torch.LongTensor(y_val_seq))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Initialize LSTM
lstm_model = TextLSTMClassifier(vocab_size, embedding_dim=128, hidden_dim=256, num_classes=5)
lstm_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)

# Train LSTM
print("\nTraining LSTM...")
num_epochs = 15
best_val_acc = 0

for epoch in range(num_epochs):
    lstm_model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = lstm_model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # Validation
    lstm_model.eval()
    correct = 0
    total = 0
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = lstm_model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
    
    val_acc = correct / total
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(lstm_model.state_dict(), 'best_lstm_model.pth')
        scheduler.step(val_loss)

print(f"✓ LSTM training complete! Best validation accuracy: {best_val_acc:.4f}")


Training LSTM for Text Features
  Total unique words: 211
  Keeping top: 211 words
✓ Vocabulary size: 213
✓ Sequences shape: (8870, 100)

Training LSTM...
Epoch 5/15 | Loss: 1.3692 | Val Acc: 0.3506
Epoch 10/15 | Loss: 1.2896 | Val Acc: 0.3489
Epoch 15/15 | Loss: 0.9138 | Val Acc: 0.3253
✓ LSTM training complete! Best validation accuracy: 0.3777


In [95]:
print("\n" + "="*50)
print("Extracting LSTM Features from Trained Model")
print("="*50)

class TextLSTMFeatureExtractor(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256):
        super(TextLSTMFeatureExtractor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=2,
                           batch_first=True, dropout=0.3, bidirectional=True)
        self.feature_dim = hidden_dim * 2
        
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        hidden_combined = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return hidden_combined

# Load trained weights
feature_extractor = TextLSTMFeatureExtractor(vocab_size, embedding_dim=128, hidden_dim=256)
feature_extractor.load_state_dict(torch.load('best_lstm_model.pth', map_location=device), strict=False)
feature_extractor.to(device)
feature_extractor.eval()

def extract_lstm_features(sequences, batch_size=128):
    features = []
    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch = torch.LongTensor(sequences[i:i+batch_size]).to(device)
            batch_features = feature_extractor(batch)
            features.append(batch_features.cpu().numpy())
    return np.vstack(features)

# Extract features
train_text_features = extract_lstm_features(train_sequences)
test_text_features = extract_lstm_features(test_sequences)
print(f"✓ LSTM text features shape: {train_text_features.shape}")


Extracting LSTM Features from Trained Model
✓ LSTM text features shape: (8870, 512)


In [96]:

print("\n" + "="*50)
print("Combining Modalities")
print("="*50)

X_multimodal_train = np.hstack([
    X_tabular_train,
    train_text_features,
    train_image_features
])

X_multimodal_test = np.hstack([
    X_tabular_test,
    test_text_features,
    test_image_features
])

print(f"✓ Multimodal train shape: {X_multimodal_train.shape}")
print(f"  - Tabular: {X_tabular_train.shape[1]}")
print(f"  - LSTM Text: {train_text_features.shape[1]}")
print(f"  - Images: {train_image_features.shape[1]}")


Combining Modalities
✓ Multimodal train shape: (8870, 1809)
  - Tabular: 17
  - LSTM Text: 512
  - Images: 1280


In [97]:

print("\n" + "="*50)
print("Training Multimodal Deep Learning Model")
print("="*50)

class MultimodalDeepLearning(nn.Module):
    def __init__(self, tabular_dim, text_dim, image_dim, num_classes=5):
        super(MultimodalDeepLearning, self).__init__()
        
        # MLP for Tabular
        self.tabular_mlp = nn.Sequential(
            nn.Linear(tabular_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32)
        )
        
        # MLP for Text
        self.text_mlp = nn.Sequential(
            nn.Linear(text_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64)
        )
        
        # MLP for Images
        self.image_mlp = nn.Sequential(
            nn.Linear(image_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128)
        )
        
        # Fusion layer
        fusion_dim = 32 + 64 + 128
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, tabular, text, image):
        tab_out = self.tabular_mlp(tabular)
        txt_out = self.text_mlp(text)
        img_out = self.image_mlp(image)
        
        combined = torch.cat([tab_out, txt_out, img_out], dim=1)
        output = self.fusion(combined)
        return output

# Split data
X_tab_train, X_tab_val, X_txt_train, X_txt_val, X_img_train, X_img_val, y_train, y_val = train_test_split(
    X_tabular_train, train_text_features, train_image_features, y,
    test_size=0.2, random_state=42, stratify=y
)

# Convert to tensors
X_tab_train = torch.FloatTensor(X_tab_train)
X_tab_val = torch.FloatTensor(X_tab_val)
X_txt_train = torch.FloatTensor(X_txt_train)
X_txt_val = torch.FloatTensor(X_txt_val)
X_img_train = torch.FloatTensor(X_img_train)
X_img_val = torch.FloatTensor(X_img_val)
y_train = torch.LongTensor(y_train)
y_val = torch.LongTensor(y_val)

# Create DataLoaders
batch_size = 128
train_dataset = TensorDataset(X_tab_train, X_txt_train, X_img_train, y_train)
val_dataset = TensorDataset(X_tab_val, X_txt_val, X_img_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Initialize model
model = MultimodalDeepLearning(
    tabular_dim=X_tabular_train.shape[1],
    text_dim=train_text_features.shape[1],
    image_dim=train_image_features.shape[1],
    num_classes=5
)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train
num_epochs = 30
best_val_f1 = 0

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for tab_batch, txt_batch, img_batch, y_batch in train_loader:
        tab_batch = tab_batch.to(device)
        txt_batch = txt_batch.to(device)
        img_batch = img_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(tab_batch, txt_batch, img_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += y_batch.size(0)
        train_correct += (predicted == y_batch).sum().item()
    
    train_acc = train_correct / train_total
    
    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for tab_batch, txt_batch, img_batch, y_batch in val_loader:
            tab_batch = tab_batch.to(device)
            txt_batch = txt_batch.to(device)
            img_batch = img_batch.to(device)
            y_batch = y_batch.to(device)
            
            outputs = model(tab_batch, txt_batch, img_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            val_total += y_batch.size(0)
            val_correct += (predicted == y_batch).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    val_acc = val_correct / val_total
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), 'best_multimodal_model.pth')
        print(f"  ✓ New best model! F1-Macro: {best_val_f1:.4f}")
    
    scheduler.step(val_loss)

print(f"\n✓ Training complete! Best F1-Macro: {best_val_f1:.4f}")


Training Multimodal Deep Learning Model
Model parameters: 476,645
  ✓ New best model! F1-Macro: 0.3120
  ✓ New best model! F1-Macro: 0.3154
  ✓ New best model! F1-Macro: 0.3206
  ✓ New best model! F1-Macro: 0.3274
Epoch 5/30 | Train Loss: 1.2109 | Train Acc: 0.4921 | Val Acc: 0.4955 | Val F1: 0.3261
  ✓ New best model! F1-Macro: 0.3329
Epoch 10/30 | Train Loss: 1.1635 | Train Acc: 0.5073 | Val Acc: 0.5000 | Val F1: 0.3313
Epoch 15/30 | Train Loss: 1.1227 | Train Acc: 0.5261 | Val Acc: 0.4961 | Val F1: 0.3293
Epoch 20/30 | Train Loss: 1.0731 | Train Acc: 0.5437 | Val Acc: 0.4899 | Val F1: 0.3249
Epoch 25/30 | Train Loss: 1.0467 | Train Acc: 0.5527 | Val Acc: 0.4876 | Val F1: 0.3289
Epoch 30/30 | Train Loss: 1.0336 | Train Acc: 0.5627 | Val Acc: 0.4938 | Val F1: 0.3333
  ✓ New best model! F1-Macro: 0.3333

✓ Training complete! Best F1-Macro: 0.3333


In [98]:

print("\n" + "="*70)
print("EVALUATING BEST MODEL")
print("="*70)

# Load best model
model.load_state_dict(torch.load('best_multimodal_model.pth'))
model.eval()

# Get predictions
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for tab_batch, txt_batch, img_batch, y_batch in val_loader:
        tab_batch = tab_batch.to(device)
        txt_batch = txt_batch.to(device)
        img_batch = img_batch.to(device)
        
        outputs = model(tab_batch, txt_batch, img_batch)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

print(f"Accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1-Macro: {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"F1-Weighted: {f1_score(all_labels, all_preds, average='weighted'):.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels, all_preds))


EVALUATING BEST MODEL
Accuracy: 0.4938
F1-Macro: 0.3333
F1-Weighted: 0.4466

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.02      0.03        63
           1       0.49      0.73      0.59       539
           2       0.00      0.00      0.00       244
           3       0.41      0.45      0.43       529
           4       0.64      0.61      0.62       399

    accuracy                           0.49      1774
   macro avg       0.41      0.36      0.33      1774
weighted avg       0.43      0.49      0.45      1774



In [99]:
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)

# Random Forest baseline
X_multi_train_np = np.hstack([X_tabular_train, train_text_features, train_image_features])
X_train_rf, X_val_rf, y_train_rf, y_val_rf = train_test_split(
    X_multi_train_np, y, test_size=0.2, random_state=42, stratify=y
)

# Apply SMOTE for Random Forest
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_rf_res, y_train_rf_res = smote.fit_resample(X_train_rf, y_train_rf)

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
rf_model.fit(X_train_rf_res, y_train_rf_res)
rf_pred = rf_model.predict(X_val_rf)
rf_f1 = f1_score(y_val_rf, rf_pred, average='macro')

print(f"Random Forest F1-Macro: {rf_f1:.4f}")
print(f"Deep Learning F1-Macro: {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"\n✅ Improvement: {f1_score(all_labels, all_preds, average='macro') - rf_f1:.4f}")


MODEL COMPARISON
Random Forest F1-Macro: 0.3374
Deep Learning F1-Macro: 0.3333

✅ Improvement: -0.0041


In [100]:

print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)

# Train simple Random Forest on multimodal features
from sklearn.ensemble import RandomForestClassifier

X_multi_train = np.hstack([X_tabular_train, train_text_features, train_image_features])
X_train_rf, X_val_rf, y_train_rf, y_val_rf = train_test_split(
    X_multi_train, y, test_size=0.2, random_state=42, stratify=y
)

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf, y_train_rf)
rf_pred = rf_model.predict(X_val_rf)
rf_f1 = f1_score(y_val_rf, rf_pred, average='macro')

print(f"\nRandom Forest F1-Macro: {rf_f1:.4f}")
print(f"Deep Learning F1-Macro: {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"\n✅ Deep Learning Improvement: {f1_score(all_labels, all_preds, average='macro') - rf_f1:.4f}")


MODEL COMPARISON

Random Forest F1-Macro: 0.3218
Deep Learning F1-Macro: 0.3333

✅ Deep Learning Improvement: 0.0115


In [101]:
# Quick sanity check - are features predictive at all?
from sklearn.ensemble import RandomForestClassifier

# Test just tabular features
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tabular_train, y)
print(f"Tabular only F1: {f1_score(y, rf.predict(X_tabular_train), average='macro'):.4f}")

# Test with cross-validation
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf, X_tabular_train, y, cv=3, scoring='f1_macro')
print(f"Cross-validation F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

Tabular only F1: 1.0000
Cross-validation F1: 0.2876 (+/- 0.0131)
